# L20 · Prompt 工程：和 AI 对话的艺术

**学习目标**
- 理解「Prompt（提示词）」如何引导大模型行为
- 掌握「角色设定 / 指令 / 示例 / 约束」四类提示技巧
- 亲手搭建一个「可定制人格」的对话模板，直观看到提示词的力量

**前置依赖**：L19（了解大模型是「下一个词预测器」）  
**预计时长**：45 分钟  
**技术栈**：纯 Python（离线模拟 LLM 行为，无需 API key，保证可运行）

---

## 概念讲解：Prompt = 给 AI 的「剧本」

大模型像个记忆力超群但没主见的天才。你给的「剧本」（Prompt）决定它演谁：

- **角色**：「你是一名严格的法律顾问」→ 它说话像律师
- **指令**：「只输出 JSON」→ 它收敛格式
- **示例（few-shot）**：给 2 个例子 → 它照着学模式
- **约束**：「不超过 20 字」→ 它自我限制

本课我们用一个**规则驱动的伪 LLM** 演示：同样的输入，不同 Prompt 产出天差地别。

## 第一步：一个「伪 LLM」——按 Prompt 规则回应

In [ ]:
def fake_llm(prompt, user_input):
    """极简规则引擎，模拟『不同 prompt 导致不同输出』"""
    p = prompt.lower()
    if "诗人" in p:
        return f"【诗】{user_input} 如风轻拂面，似梦落心间。"
    if "律师" in p:
        return f"【法律意见】关于「{user_input}」，建议保留证据并咨询执业律师。"
    if "json" in p:
        return f'{{"input": "{user_input}", "len": {len(user_input)}}}'
    return f"（普通）你说的是：{user_input}"

topic = "今天天气真好"
print("普通：", fake_llm("你是个助手", topic))
print("诗人：", fake_llm("你是个诗人", topic))
print("律师：", fake_llm("你是个律师", topic))

## 第二步：few-shot（给示例让模型照学）

In [ ]:
def fake_translate(prompt_examples, word):
    """看示例学会了『中→英』映射规则"""
    mapping = {}
    for line in prompt_examples.strip().split("\n"):
        if "→" in line:
            zh, en = line.split("→")
            mapping[zh.strip()] = en.strip()
    return mapping.get(word, f"<未知:{word}>")

ex = "猫 → cat\n狗 → dog\n鸟 → bird"
print("鸟 →", fake_translate(ex, "鸟"))
print("猫 →", fake_translate(ex, "猫"))

# 🎯 AHA 顿悟单元格：一键切换 AI 人格

运行下面代码。你会得到一个**「人格切换器」**：同一句话，选择不同角色 Prompt，
AI 的回应风格瞬间从「诗人」变「律师」变「JSON 机器」。改 `user_input` 再试。

> 这就是 Prompt 工程的威力：**不改模型一行参数，只改剧本，AI 就能千变万化**。
> 它是今天调用 GPT/Claude 最省钱、最快的「编程方式」。

In [ ]:
# ===== 运行我！切换 role 看不同人格 =====
def fake_llm(prompt, user_input):
    p = prompt.lower()
    if "诗人" in p:
        return f"【诗】{user_input} 如风轻拂面，似梦落心间。"
    if "律师" in p:
        return f"【法律意见】关于「{user_input}」，建议保留证据并咨询执业律师。"
    if "json" in p:
        return f'{{"input": "{user_input}", "len": {len(user_input)}}}'
    if "客服" in p:
        return f"【客服】亲，您提到的「{user_input}」我们已记录，2小时内回复您哦~ 💁"
    return f"（普通）你说的是：{user_input}"

user_input = "我的订单还没发货"
roles = {
    "诗人": "你是个浪漫的诗人",
    "律师": "你是个严谨的律师",
    "客服": "你是个热情的电商客服",
    "JSON机器": "只输出 JSON",
}
print(f"  💬 用户说：{user_input}\n")
print("  " + "=" * 42)
for name, prompt in roles.items():
    print(f"  🎭 [{name}] {fake_llm(prompt, user_input)}")
print("  " + "=" * 42)
print("  ✨ 同一个输入，四种人格 —— 全靠 Prompt 决定！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：学员易误以为「伪 LLM」就是真 LLM；需在讲解中明确这是「行为模拟」，真 LLM 是概率预测，但 Prompt 机制一致。  
**易错点**：规则匹配误触发（如「JSON机器」需精确匹配）。  
**AHA 机制**：人格切换器直观展示 Prompt 决定性，强「我控制了 AI」掌控感，零依赖保证运行。  
**衔接**：L21 函数调用（让 AI 不只说话还能动手）；L22 RAG（给 AI 外挂知识）；L24 微调（改参数而非 Prompt）。  
**真 LLM 衔接**：在备课笔记注明，若学员有 API key，可把 `fake_llm` 换成 `openai` 调用，给出伪代码。

# 📚 作业 / 下一步

1. 给 `fake_llm` 加一个「科学家」人格（用数据口吻回答）。
2. 思考：如果规则不够覆盖所有输入怎么办？（引出真 LLM 的泛化能力）
3. 下一课 **L21 函数调用：让 AI 动手做事** —— 让 AI 不只是聊天，还能调用你的代码/工具。